# Task 3: Evaluation and Failure Analysis
Each step and function is in its own cell so you can inspect and run it cleanly.

### Cell 1: Unzip inputs

In [ ]:
!unzip -q -o "/content/egg.v3i.folder.zip" -d /content/
!unzip -q -o "/content/task1_outputs.zip" -d /content/task1_outputs
!unzip -q -o "/content/egg_task2_out.zip" -d /content/task2_outputs

### Cell 2: Imports, paths, images, and answer-key masks

In [ ]:
import json, re, time
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image
from scipy import ndimage as ndi
from scipy.optimize import linear_sum_assignment

DATA = Path("/content/egg.v3i.folder")                                          # the photos
SAM_DIR = next(Path("/content/task1_outputs").rglob("sam_masks"))               # Task 1 masks = the answer key
TASK2_DIR = next(Path("/content/task2_outputs").rglob("final_config.json")).parent   # frozen config + split from Task 2
out = Path("/content/task3_outputs")
out.mkdir(exist_ok=True)

imgs = sorted(p for p in DATA.rglob("*") if p.suffix.lower() in {".jpg", ".jpeg", ".png"})
mask_of = {p.stem.removesuffix("_mask"): p for p in SAM_DIR.rglob("*.png")}      # image name -> its answer-key mask
missing = [p.stem for p in imgs if p.stem not in mask_of]
print(len(imgs), "images |", len(mask_of), "masks | images without a mask:", len(missing))
assert imgs and not missing, "images or masks are missing"

### Cell 3: `get(i)` — Load one photo with its answer key

In [ ]:
cache = {}

def get(i):
    """Return (photo as BGR, answer key as True/False image) for image number i."""
    if i not in cache:
        photo = cv2.imread(str(imgs[i]))
        raw = np.array(Image.open(mask_of[imgs[i].stem]))
        key = raw if raw.ndim == 2 else raw.max(axis=2)       # works for 0/255, label images and RGB masks
        assert key.shape == photo.shape[:2], f"mask and photo sizes differ for {imgs[i].name}"
        cache[i] = (photo, key > 0)
    return cache[i]

### Cell 4: Default parameters and geometry helpers (`kernel`, `largest_contour`, `solidity`)

In [ ]:
DEFAULTS = dict(denoise="gauss", mode="otsu_gray", s_max=65, v_min=150, l_min=140, ab_tol=22, skin=False,
                morph=False, k_open=5, k_close=15, select="largest", min_area=0.005, max_area=0.98,
                min_solid=0.9, refine=None, gc_margin=0.08, gc_iters=4)

def kernel(size):
    return cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (size, size))

def largest_contour(blob):
    contours = cv2.findContours(blob, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)[0]
    return max(contours, key=cv2.contourArea)

def solidity(blob):
    """Blob area divided by the area of its convex hull. A smooth egg is close to 1."""
    contour = largest_contour(blob)
    hull_area = cv2.contourArea(cv2.convexHull(contour))
    return cv2.contourArea(contour) / hull_area if hull_area else 0.0

### Cell 5: `egg_score(blob)` — Shape & position score for blob selection

In [ ]:
def egg_score(blob):
    """How egg-like is this blob? Convex, ellipse-shaped, near the centre and big all push the score up."""
    contour = largest_contour(blob)
    area = cv2.contourArea(contour)
    height, width = blob.shape
    if area < 1 or len(contour) < 5:
        return 0.0
    (cx, cy), (axis1, axis2), _ = cv2.fitEllipse(contour)
    ellipse_area = np.pi * axis1 * axis2 / 4
    ellipse_fit = min(area, ellipse_area) / max(area, ellipse_area, 1)
    off_centre = np.hypot(cx - width / 2, cy - height / 2) / np.hypot(width / 2, height / 2)
    return solidity(blob) * ellipse_fit * (1 - 0.5 * off_centre) * np.sqrt(area / (height * width))

### Cell 6: `grabcut_refine` and `denoise`

In [ ]:
def grabcut_refine(img, mask, iterations=3, ring=15):
    """Pull the edge onto the real boundary: inside = sure egg, a ring around it = unsure, outside = sure background."""
    state = np.full(mask.shape, cv2.GC_BGD, np.uint8)
    state[cv2.dilate(mask, kernel(2 * ring + 1)) > 0] = cv2.GC_PR_BGD
    state[mask > 0] = cv2.GC_PR_FGD
    state[cv2.erode(mask, kernel(2 * ring + 1)) > 0] = cv2.GC_FGD
    cv2.grabCut(img, state, None, np.zeros((1, 65)), np.zeros((1, 65)), iterations, cv2.GC_INIT_WITH_MASK)
    return np.where((state == cv2.GC_FGD) | (state == cv2.GC_PR_FGD), 255, 0).astype(np.uint8)

def denoise(bgr, kind):
    if kind == "gauss":
        return cv2.GaussianBlur(bgr, (5, 5), 0)
    if kind == "bilateral":
        return cv2.bilateralFilter(bgr, 9, 50, 50)
    return bgr

### Cell 7: `rough_mask(img, cfg)` — Initial binary thresholding / GrabCut

In [ ]:
def rough_mask(img, cfg):
    """Step 1: a first black/white guess (white = maybe egg)."""
    mode = cfg["mode"]
    if mode == "otsu_gray":                                   # brightness only
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        return cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)[1]
    if mode == "grabcut":                                     # no colour rule: assume the egg is big and roughly central
        height, width = img.shape[:2]
        margin = int(cfg["gc_margin"] * min(height, width))
        state = np.zeros((height, width), np.uint8)
        box = (margin, margin, width - 2 * margin, height - 2 * margin)
        cv2.grabCut(img, state, box, np.zeros((1, 65)), np.zeros((1, 65)), cfg["gc_iters"], cv2.GC_INIT_WITH_RECT)
        return np.where((state == cv2.GC_FGD) | (state == cv2.GC_PR_FGD), 255, 0).astype(np.uint8)
    # "hsv" / "hsv_lab": a white egg is bright (high V) and colourless (low S)
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    mask = cv2.inRange(hsv, (0, 0, cfg["v_min"]), (180, cfg["s_max"], 255))
    if mode == "hsv_lab":                                     # also require neutral a*, b* in LAB
        lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
        low = (cfg["l_min"], 128 - cfg["ab_tol"], 128 - cfg["ab_tol"])
        high = (255, 128 + cfg["ab_tol"], 128 + cfg["ab_tol"])
        mask &= cv2.inRange(lab, low, high)
    return mask

### Cell 8: `pick_blobs` and `draw_blob`

In [ ]:
def pick_blobs(mask, cfg):
    """Step 3: from all white regions, keep the one(s) that should be the egg."""
    count, labels, stats, _ = cv2.connectedComponentsWithStats(mask)
    areas = stats[:, cv2.CC_STAT_AREA]
    blob = lambda k: (labels == k).astype(np.uint8)
    size_ok = [k for k in range(1, count) if cfg["min_area"] <= areas[k] / mask.size <= cfg["max_area"]]

    if cfg["select"] == "largest":
        chosen = [1 + int(areas[1:].argmax())] if count > 1 else []
    elif cfg["select"] == "score":                            # the single most egg-like blob
        scores = {k: egg_score(blob(k)) for k in size_ok}
        chosen = [max(scores, key=scores.get)] if scores else []
    else:                                                     # "shape": every smooth, convex blob (handles several eggs)
        chosen = [k for k in size_ok if solidity(blob(k)) >= cfg["min_solid"]]
    return [blob(k) for k in chosen]

def draw_blob(canvas, blob, method):
    """Step 4: copy the blob onto the canvas, optionally replacing it by its convex hull or a fitted ellipse."""
    hull = cv2.convexHull(largest_contour(blob))
    if method == "hull":
        cv2.fillPoly(canvas, [hull], 255)
    elif method == "ellipse" and len(hull) >= 5:
        cv2.ellipse(canvas, cv2.fitEllipse(hull), 255, -1)
    else:
        canvas[blob > 0] = 255

### Cell 9: `seg(bgr, cfg)` — The frozen Task 2 algorithm

In [ ]:
def seg(bgr, cfg):
    """The frozen Task 2 algorithm. Returns a 0/255 mask."""
    cv2.setRNGSeed(0)            # GrabCut starts from random k-means centres; seeding makes every run give the same mask
    img = denoise(bgr, cfg["denoise"])
    mask = rough_mask(img, cfg)

    if cfg["skin"]:                                           # remove skin-coloured pixels (YCrCb range)
        skin = cv2.inRange(cv2.cvtColor(img, cv2.COLOR_BGR2YCrCb), (0, 133, 77), (255, 173, 127))
        mask[skin > 0] = 0
    if cfg["morph"]:                                          # open = remove specks, close = fill small holes
        mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel(cfg["k_open"]))
        mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel(cfg["k_close"]))
    mask = (ndi.binary_fill_holes(mask > 0) * 255).astype(np.uint8)

    result = np.zeros_like(mask)
    for blob in pick_blobs(mask, cfg):
        draw_blob(result, blob, cfg["refine"])
    if cfg["refine"] == "grabcut" and result.any():
        result = grabcut_refine(img, result)
    return result

### Cell 10: Load frozen config and saved split (`final_config.json` + `split.json`)

In [ ]:
raw_cfg = json.load(open(TASK2_DIR / "final_config.json"))
cfg3 = {**DEFAULTS, **raw_cfg.get("config", raw_cfg)}                           # handles both flat and nested config dicts
split_path = next(Path("/content/task2_outputs").rglob("split.json"))
split = json.load(open(split_path))                                             # the SAME split as Task 2
name_to_index = {p.name: i for i, p in enumerate(imgs)}
dev_i = [name_to_index[n] for n in split["dev"]]
eval_i = [name_to_index[n] for n in split["eval"]]
assert dev_i and eval_i and not (set(dev_i) & set(eval_i)), "dev and eval overlap or empty"
print("frozen config:", cfg3)
print("dev:", len(dev_i), "| eval:", len(eval_i))

### Cell 11: Pixel & boundary metric functions (`overlap_scores`, `edge_pixels`, `boundary_scores`)

In [ ]:
def overlap_scores(pred, key):
    """Pixel-level scores. pred and key are True/False images."""
    inter = (pred & key).sum()
    union = (pred | key).sum()
    return dict(
        iou=inter / union if union else 1.0,
        dice=2 * inter / (pred.sum() + key.sum()) if union else 1.0,
        prec=inter / pred.sum() if pred.sum() else float(not key.any()),   # of what I painted, how much is egg
        rec=inter / key.sum() if key.sum() else float(not pred.any()),     # of the egg, how much did I paint
    )

def edge_pixels(mask):
    """The outline of a mask. The picture frame is not an object edge, so pixels outside the frame count as mask."""
    shrunk = cv2.erode(mask.astype(np.uint8), np.ones((3, 3), np.uint8), borderType=cv2.BORDER_CONSTANT, borderValue=255)
    return mask & ~shrunk.astype(bool)

def boundary_scores(pred, key, tolerance=3):
    """Boundary F1 (outline pixels within 3 px of the other outline) and HD95 (95th percentile outline distance, in pixels)."""
    pred_edge, key_edge = edge_pixels(pred), edge_pixels(key)
    if pred_edge.any() and key_edge.any():
        dist_to_key = ndi.distance_transform_edt(~key_edge)
        dist_to_pred = ndi.distance_transform_edt(~pred_edge)
        edge_precision = (dist_to_key[pred_edge] <= tolerance).mean()
        edge_recall = (dist_to_pred[key_edge] <= tolerance).mean()
        f1 = 2 * edge_precision * edge_recall / (edge_precision + edge_recall) if edge_precision + edge_recall else 0.0
        hd95 = float(np.percentile(np.r_[dist_to_key[pred_edge], dist_to_pred[key_edge]], 95))
        return f1, hd95
    both_empty = not pred_edge.any() and not key_edge.any()
    return float(both_empty), np.nan          # HD95 is undefined when one outline is missing

### Cell 12: Instance matching functions (`split_into_eggs`, `key_eggs`, `count_matches`)

In [ ]:
def split_into_eggs(mask, min_fraction=0.002):
    """Separate blobs of a mask, ignoring specks smaller than 0.2% of the image."""
    count, labels, stats, _ = cv2.connectedComponentsWithStats(mask.astype(np.uint8))
    return [labels == k for k in range(1, count) if stats[k, cv2.CC_STAT_AREA] >= min_fraction * mask.size]

def key_eggs(i, key):
    """Eggs in the answer key. If the Task 1 mask stores one label id per egg, use those so touching eggs stay separate."""
    raw = np.array(Image.open(mask_of[imgs[i].stem]))
    if raw.ndim == 2 and 1 < raw.max() < 255:
        eggs = [raw == k for k in np.unique(raw) if k > 0]
        eggs = [e for e in eggs if e.mean() >= 0.002]
        if eggs:
            return eggs
    return split_into_eggs(key)

def count_matches(pred_eggs, true_eggs):
    """Pair predicted and true eggs one-to-one (Hungarian matching); a pair counts if IoU >= 0.5."""
    if not pred_eggs or not true_eggs:
        return 0
    iou = np.array([[(p & t).sum() / (p | t).sum() for t in true_eggs] for p in pred_eggs])
    rows, cols = linear_sum_assignment(-iou)
    return int((iou[rows, cols] >= 0.5).sum())

### Cell 13: `score_images(indices, cfg)` — Run evaluation over a split

In [ ]:
def score_images(indices, cfg):
    rows = []
    for i in indices:
        image, key = get(i)
        start = time.time()
        pred = seg(image, cfg) > 127
        milliseconds = (time.time() - start) * 1000

        f1, hd95 = boundary_scores(pred, key)
        pred_eggs, true_eggs = split_into_eggs(pred), key_eggs(i, key)
        brightness = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)[..., 2]

        rows.append(dict(
            image=imgs[i].stem, ms=milliseconds, empty_key=not key.any(),
            pred_area=pred.mean(), gt_area=key.mean(),
            egg_V=brightness[key].mean() if key.any() else np.nan,
            bg_V=brightness[~key].mean() if (~key).any() else np.nan,
            **overlap_scores(pred, key), bf1=f1, hd95=hd95,
            n_gt=len(true_eggs), n_pred=len(pred_eggs), tp=count_matches(pred_eggs, true_eggs),
        ))
    return pd.DataFrame(rows)

### Cell 14: Score eval once and save CSVs

In [ ]:
# Score every image once with the frozen config. Eval is scored here and nowhere else.
ev = score_images(eval_i, cfg3)
dv = score_images(dev_i, cfg3)

# images whose Task 1 mask is empty cannot be scored fairly, so they are kept in the csv but left out of the averages
ev_ok = ev[~ev.empty_key].copy()
dv_ok = dv[~dv.empty_key].copy()

ev.to_csv(out / "eval_metrics.csv", index=False)
dv.to_csv(out / "dev_metrics.csv", index=False)
print("scored", len(ev), "eval and", len(dv), "dev images;", int(ev.empty_key.sum()), "eval and", int(dv.empty_key.sum()), "dev have an empty key")

### Cell 15: Summary table and 95% bootstrap confidence interval

In [ ]:
metrics = ["iou", "dice", "prec", "rec", "bf1", "hd95"]

def describe(df):
    return df[metrics].agg(["mean", "median", "std", "min"]).T.round(3)

summary = pd.concat({"dev": describe(dv_ok), "eval": describe(ev_ok)}, axis=1)
summary.to_csv(out / "summary.csv")
print(summary)
print(f"\nHD95 exists only where both outlines exist: {int(ev_ok.hd95.notna().sum())} of {len(ev_ok)} eval images")

# how much could the eval mean IoU move just by chance? resample the eval images with replacement 1000 times
rng = np.random.default_rng(0)
iou = ev_ok.iou.values
means = [rng.choice(iou, size=len(iou)).mean() for _ in range(1000)]
ci_low, ci_high = np.percentile(means, [2.5, 97.5])
print(f"\neval mean IoU {iou.mean():.3f}  (95% interval {ci_low:.3f} to {ci_high:.3f}, n={len(iou)})")
print(f"dev - eval gap: {dv_ok.iou.mean() - iou.mean():+.3f}")

### Cell 16: `instance_scores(df)` — Egg-by-egg scores, exclusions, and runtime

In [ ]:
def instance_scores(df):
    """Egg-by-egg scores: a predicted egg is right if it overlaps a true egg with IoU >= 0.5."""
    tp, predicted, actual = df.tp.sum(), df.n_pred.sum(), df.n_gt.sum()
    precision = tp / predicted if predicted else 0.0
    recall = tp / actual if actual else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    count_error = (df.n_pred - df.n_gt).abs().mean()
    return dict(precision=round(float(precision), 3), recall=round(float(recall), 3),
                f1=round(float(f1), 3), count_error=round(float(count_error), 3))

print("instance level, eval:", instance_scores(ev_ok))
print("instance level, dev :", instance_scores(dv_ok))
predicted_on_empty = int((ev[ev.empty_key].pred_area > 0).sum())
print(f"excluded (empty Task 1 mask): {int(ev.empty_key.sum())} eval images; the algorithm still painted something on {predicted_on_empty} of them")
print(f"runtime: {ev.ms.mean():.0f} ms per image")

### Cell 17: Quantitative plots (`fig_quant.png`)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

# 1. how are IoU scores spread? (a mean can hide a split into "good" and "failed")
axes[0, 0].hist([dv_ok.iou, ev_ok.iou], bins=10, range=(0, 1), label=["dev", "eval"])
axes[0, 0].set(title="IoU distribution", xlabel="IoU")
axes[0, 0].legend()

# 2. spread of each metric on eval
axes[0, 1].boxplot([ev_ok[m].dropna() for m in ["iou", "dice", "prec", "rec", "bf1"]])
axes[0, 1].set_xticklabels(["IoU", "Dice", "precision", "recall", "BF1"])
axes[0, 1].set_title("eval metrics")

# 3. low precision = painted too much, low recall = missed part of the egg
axes[0, 2].scatter(ev_ok.rec, ev_ok.prec, c=ev_ok.iou, vmin=0, vmax=1)
axes[0, 2].set(title="precision vs recall", xlabel="recall", ylabel="precision")

# 4. does egg size matter?
axes[1, 0].scatter(ev_ok.gt_area, ev_ok.iou)
axes[1, 0].set(title="IoU vs egg size", xlabel="egg area / image area", ylabel="IoU")

# 5. does contrast matter? (left of the red line = egg darker than its background)
axes[1, 1].scatter(ev_ok.egg_V - ev_ok.bg_V, ev_ok.iou)
axes[1, 1].axvline(0, color="red", linestyle="--")
axes[1, 1].set(title="IoU vs egg-minus-background brightness", xlabel="egg V - background V", ylabel="IoU")

# 6. one egg versus several
by_count = ev_ok.groupby("n_gt").iou.apply(list)
axes[1, 2].boxplot(by_count.tolist())
axes[1, 2].set_xticklabels([f"{k} egg(s)\n(n={len(v)})" for k, v in by_count.items()])
axes[1, 2].set_title("IoU vs number of eggs in the key")

plt.tight_layout()
plt.savefig(out / "fig_quant.png", dpi=110)
plt.show()

### Cell 18: The 8 worst eval images (`fig_worst.png`)

In [ ]:
# The 8 worst eval images. green = correct, red = painted but not egg, blue = egg we missed.
worst = ev_ok.nsmallest(8, "iou")
fig, axes = plt.subplots(len(worst), 4, figsize=(13, 3 * len(worst)), squeeze=False)

for row_number, (_, row) in enumerate(worst.iterrows()):
    i = next(j for j in eval_i if imgs[j].stem == row.image)
    photo, key = get(i)
    pred = seg(photo, cfg3) > 127

    coloured = photo[:, :, ::-1].copy()                      # BGR -> RGB for display
    for region, colour in [(key & pred, (0, 255, 0)), (pred & ~key, (255, 0, 0)), (key & ~pred, (0, 0, 255))]:
        coloured[region] = (coloured[region] * 0.4 + np.array(colour) * 0.6).astype(np.uint8)

    panels = [(photo[:, :, ::-1], row.image[:20]),
              (key, "answer key"),
              (pred, f"prediction  P={row.prec:.2f}  R={row.rec:.2f}"),
              (coloured, f"IoU {row.iou:.2f}")]
    for ax, (picture, title) in zip(axes[row_number], panels):
        if picture.ndim == 2:
            ax.imshow(picture, cmap="gray", vmin=0, vmax=1)  # fixed range so an all-True mask is not drawn black
        else:
            ax.imshow(picture)
        ax.set_title(title, fontsize=8)
        ax.axis("off")

plt.tight_layout()
plt.savefig(out / "fig_worst.png", dpi=110)
plt.show()

### Cell 19: `error_shape(row)` — Automatic failure categorization

In [ ]:
def error_shape(row):
    """What KIND of mistake is it? (the cause comes later, from your own judgement)"""
    if row.iou >= 0.85:
        return "good"
    if row.pred_area == 0:
        return "missed"
    if row.iou < 0.1:
        return "wrong_object"
    if row.prec < 0.7 <= row.rec:
        return "over_segmented"          # painted too much
    if row.rec < 0.7 <= row.prec:
        return "under_segmented"         # missed part of the egg
    return "mixed"

ev_ok["error_shape"] = ev_ok.apply(error_shape, axis=1)
print(ev_ok.error_shape.value_counts().to_string())
print("\nthe 12 worst, with context:")
print(ev_ok.nsmallest(12, "iou")[["image", "iou", "prec", "rec", "egg_V", "bg_V", "n_gt", "error_shape"]].round(2).to_string(index=False))

### Cell 20: Your manual `CAUSE` labels and taxonomy chart (`fig_taxonomy.png`)

In [ ]:
# Fill this in AFTER looking at the gallery. Key = short image id, value = your own words for WHY it failed.
CAUSE = {
    # "not_damaged_108": "dish picked as the egg",
}

short_id = lambda name: re.sub(r"(_jpg)?\.rf\..*$", "", name)
unmatched = [k for k in CAUSE if k not in set(ev_ok.image.map(short_id))]
assert not unmatched, f"these CAUSE keys match no eval image: {unmatched}"

ev_ok["cause"] = ev_ok.image.map(lambda name: CAUSE.get(short_id(name)))
ev_ok.to_csv(out / "eval_usable_taxonomy.csv", index=False)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
ev_ok.error_shape.value_counts().plot.barh(ax=axes[0], title="kind of mistake (automatic)")
if ev_ok.cause.notna().any():
    ev_ok.cause.dropna().value_counts().plot.barh(ax=axes[1], title="cause (my labels)")
else:
    axes[1].text(0.5, 0.5, "fill in CAUSE and re-run", ha="center")
plt.tight_layout()
plt.savefig(out / "fig_taxonomy.png", dpi=110)
plt.show()

### Cell 21: Key numbers for the report

In [ ]:
print(f"""Task 3 numbers for the report
- frozen config: {cfg3}
- eval (n={len(ev_ok)}): mean IoU {ev_ok.iou.mean():.3f} (95% interval {ci_low:.3f}-{ci_high:.3f}), median {ev_ok.iou.median():.3f}
  Dice {ev_ok.dice.mean():.3f}, precision {ev_ok.prec.mean():.3f}, recall {ev_ok.rec.mean():.3f}, boundary F1 {ev_ok.bf1.mean():.3f}
- dev (n={len(dv_ok)}): mean IoU {dv_ok.iou.mean():.3f}  -> dev - eval gap {dv_ok.iou.mean() - ev_ok.iou.mean():+.3f}
- egg by egg: {instance_scores(ev_ok)}
- IoU >= 0.85 on {int((ev_ok.iou >= 0.85).sum())} of {len(ev_ok)}; IoU < 0.1 on {int((ev_ok.iou < 0.1).sum())}
- excluded: {int(ev.empty_key.sum())} eval image(s) with an empty Task 1 mask
- runtime: {ev.ms.mean():.0f} ms per image
To write yourself: what the numbers mean; the top causes with example images; what to try next; limits.""")